# Voxtral ASR Baseline Setup
This notebook sets up the `mistralai/Voxtral-Mini-4B-Realtime-2602` model using `Transformers` and exposes it via `ngrok` for remote access.

## 1. Environment Setup

In [ ]:
import sys
import os
import subprocess
import importlib.metadata as metadata


def installed_version(name: str) -> str | None:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


TARGET_PACKAGES = ["fastapi", "uvicorn", "transformers", "mistral-common", "bitsandbytes", "torch"]
before_versions = {name: installed_version(name) for name in TARGET_PACKAGES}

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Detected Google Colab environment. Installing Voxtral serving stack...")
    !python -m pip install -U pip

    !python -m pip uninstall -y vllm compressed-tensors

    # websockets must stay <16.0 so that google-adk and gradio-client on Colab don't break
    !python -m pip install -U "transformers>=5.2.0" "mistral-common[audio]>=1.9.1" fastapi uvicorn bitsandbytes librosa "websockets>=15.0.1,<16.0" soundfile soxr pyngrok python-dotenv "requests==2.32.4" packaging

    after_versions = {name: installed_version(name) for name in TARGET_PACKAGES}
    print("\nInstalled versions:")
    for pkg in TARGET_PACKAGES:
        print(f"  {pkg}: {after_versions[pkg] or 'not found'}")

    changed = before_versions != after_versions
    missing_after = [name for name, value in after_versions.items() if value is None and name != 'torch']
    if missing_after:
        raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

    if changed:
        raise SystemExit(
            'The serving stack changed in this runtime. Restart the runtime now, then run all cells again '
            'to avoid stale imports and false dependency mismatches.'
        )

    print("\nServing stack already matched the required versions. No restart needed.")
else:
    print("Running in local environment. Ensure dependencies are installed via requirements.txt")


## 1.1. Hugging Face Login
Most Mistral models require authentication. Generate a token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
from dotenv import load_dotenv
import os
from huggingface_hub import login
import sys

# Load local .env if available
load_dotenv()

token = None
try:
    # Try to load from Colab Secrets if in Colab
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    # Fallback to local .env
    token = os.getenv("HF_TOKEN")

if token:
    login(token=token)
    print("Authenticated with Hugging Face.")
else:
    print("HF_TOKEN not found in Colab Secrets or .env. Using interactive login...")
    !huggingface-cli login

In [ ]:
import importlib.metadata as metadata
import sys
from packaging.version import Version


def installed_version(name: str) -> str | None:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


versions = {name: installed_version(name) for name in ["transformers", "mistral-common", "torch"]}
print("Resolved serving stack:")
for name, value in versions.items():
    print(f"  {name}: {value or 'missing'}")

missing = [name for name, value in versions.items() if value is None and name != 'torch']
if missing:
    raise RuntimeError(f"Missing required packages for serving: {', '.join(missing)}")

transformers_version = Version(versions['transformers'])
mistral_common_version = Version(versions['mistral-common'])

if transformers_version < Version('5.2.0'):
    raise RuntimeError(f"transformers>=5.2.0 is required, found {transformers_version}")

if mistral_common_version < Version('1.9.1'):
    raise RuntimeError(f"mistral-common>=1.9.1 is required, found {mistral_common_version}")

try:
    from transformers import VoxtralRealtimeForConditionalGeneration  # noqa: F401
    from mistral_common.tokens.tokenizers.audio import Audio  # noqa: F401
except Exception as exc:
    raise RuntimeError(
        'Required classes not found. Ensure transformers>=5.2.0 and mistral-common[audio]>=1.9.1 are installed. '
        'Reinstall the Voxtral stack before launching the server.'
    ) from exc

print("Serving stack preflight passed for Transformer backend.")


## 1.2. Prepare Repository Source on Colab
The server script lives in this repo, so Colab must clone the source before launching the FastAPI process.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.getenv("VOXTRAL_REPO_URL", "https://github.com/TCTri205/VJ.git")
REPO_BRANCH = os.getenv("VOXTRAL_REPO_BRANCH", "Voxtral")
REPO_SUBDIR = os.getenv("VOXTRAL_REPO_SUBDIR", "Voxtral")
CLONE_ROOT = Path("/content/voxtral-src")
REPO_ROOT = CLONE_ROOT / "repo"

if 'google.colab' in sys.modules:
    CLONE_ROOT.mkdir(parents=True, exist_ok=True)
    if REPO_ROOT.exists():
        print(f"Repository already present at: {REPO_ROOT}. Pulling latest changes...")
        subprocess.run(["git", "reset", "--hard", f"origin/{REPO_BRANCH}"], cwd=str(REPO_ROOT), check=True)
        subprocess.run(["git", "pull", "origin", REPO_BRANCH], cwd=str(REPO_ROOT), check=True)
    else:
        gh_token = None
        try:
            from google.colab import userdata
            gh_token = userdata.get('GITHUB_TOKEN')
        except (ImportError, Exception):
            gh_token = os.getenv("GITHUB_TOKEN")

        if gh_token:
            # Inject token into the URL for authenticated clone
            authenticated_url = REPO_URL.replace("https://", f"https://{gh_token}@")
        else:
            authenticated_url = REPO_URL
            print("GITHUB_TOKEN not found. Attempting public clone...")

        clone_cmd = ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH]
        clone_cmd.extend([authenticated_url, str(REPO_ROOT)])
        print(f"Cloning repository: {REPO_URL} (Branch: {REPO_BRANCH})")
        subprocess.run(clone_cmd, check=True)
    PROJECT_DIR = REPO_ROOT / REPO_SUBDIR
else:
    PROJECT_DIR = Path.cwd()
    print(f"Running outside Colab. Using local project directory: {PROJECT_DIR}")

if not PROJECT_DIR.exists():
    raise RuntimeError(f"Project subdirectory not found: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
SERVER_SCRIPT_PATH = (PROJECT_DIR / "voxtral_server_transformers.py").resolve()
if not SERVER_SCRIPT_PATH.exists():
    raise RuntimeError(
        f"Server script not found at {SERVER_SCRIPT_PATH}. Check REPO_URL/REPO_SUBDIR before launching the server."
    )

print(f"Using project directory: {PROJECT_DIR}")
print(f"Resolved server script : {SERVER_SCRIPT_PATH}")

# --- Revision / fingerprint ---
# Print branch, commit SHA, and script hash so we can confirm Colab is running
# the intended code and not a stale/cached version.
try:
    branch = subprocess.check_output(
        ["git", "rev-parse", "--abbrev-ref", "HEAD"],
        cwd=str(REPO_ROOT), stderr=subprocess.DEVNULL
    ).decode().strip()
    sha = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=str(REPO_ROOT), stderr=subprocess.DEVNULL
    ).decode().strip()
    print(f"Git branch : {branch}")
    print(f"Git SHA    : {sha}")
except Exception:
    print("(git not available — using file hash as fingerprint)")

import hashlib
try:
    with open(SERVER_SCRIPT_PATH, "rb") as fh:
        script_hash = hashlib.sha1(fh.read()).hexdigest()[:12]
    print(f"Script SHA1: {script_hash}  ({SERVER_SCRIPT_PATH.name})")
except Exception as e:
    print(f"Could not hash script: {e}")


## 2. Ngrok Tunnel Configuration
Enter your ngrok authtoken from [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken).

In [ ]:
from pyngrok import ngrok
import getpass
import os

# Try to load Ngrok token from Colab Secrets or Env
ngrok_token = None
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
    if ngrok_token:
        print("Loaded Ngrok Authtoken from Colab Secrets.")
except (ImportError, Exception):
    ngrok_token = os.getenv("NGROK_AUTHTOKEN")

if not ngrok_token:
    print("NGROK_AUTHTOKEN not found in Colab Secrets or .env.")
    print("Enter your Ngrok Authtoken manually (from dashboard.ngrok.com):")
    ngrok_token = getpass.getpass()

ngrok.set_auth_token(ngrok_token)

# Open a HTTP tunnel on port 8000
public_url = ngrok.connect(8000, "http")
print(f"Ngrok Tunnel is active at: {public_url}")
print("Use this URL as the --host argument in run_asr.py")

## 3. Launch Transformers Server (Background)
We clone the repo source first, then launch the FastAPI server from the resolved script path and wait for it to be ready.

In [ ]:
import os
import subprocess
import time
from pathlib import Path

import requests
import torch


def read_server_log() -> str:
    try:
        with open('/tmp/voxtral_server.log', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return ''


HAS_GPU = torch.cuda.is_available()
print(f"GPU Available: {HAS_GPU}")

model_id = "mistralai/Voxtral-Mini-4B-Realtime-2602"
server_script = Path(globals().get("SERVER_SCRIPT_PATH", Path.cwd() / "voxtral_server_transformers.py")).resolve()
if not server_script.exists():
    raise RuntimeError(
        f"Cannot launch server because the script is missing: {server_script}. Run the repository setup cell first."
    )

launch_env = os.environ.copy()
launch_cwd = str(server_script.parent)
args = ["--model", model_id, "--port", "8000"]

if HAS_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    print(f"GPU detected: {gpu_name} ({major}.{minor})")
    if major < 8:
        print("Turing GPU (T4) detected. Using 4-bit quantization for VRAM safety.")
        args.append("--load-in-4bit")

command = ["python", str(server_script)] + args

print(f"Launch command: {' '.join(command)}")
print(f"Launch cwd: {launch_cwd}")
log_file = open("/tmp/voxtral_server.log", "w", encoding="utf-8")
process = subprocess.Popen(command, stdout=log_file, stderr=log_file, env=launch_env, cwd=launch_cwd)
print("Starting Voxtral Transformers server... (Log: /tmp/voxtral_server.log)")

max_wait = 60 * 15
waited = 0
while waited < max_wait:
    if process.poll() is not None:
        print(f'\nServer exited unexpectedly with code: {process.returncode}')
        print("--- Log History ---")
        print(read_server_log())
        break
    try:
        response = requests.get("http://localhost:8000/v1/models", timeout=2)
        if response.status_code == 200:
            print("Server is ready!")
            break
    except Exception:
        pass
    time.sleep(10)
    waited += 10
    if waited % 60 == 0:
        log_text = read_server_log()
        print(f"Wait {waited}s... Last log line: {log_text.splitlines()[-1] if log_text.splitlines() else 'None'}")
else:
    print("Timeout: Server not ready after 15 minutes.")


## 3.1. View Server Log
Run this cell at any time to see what the server has printed to `/tmp/voxtral_server.log`.

If the server shows `[startup] fingerprint:` with a git SHA, it is running the correct revision. If `/v1/models` passed but WebSocket realtime fails, check here for tracebacks.

In [ ]:
# ── View server log ──────────────────────────────────────────────────────────
# Run this cell to see the last N lines of the server log.
# Useful for diagnosing timeout or inference errors without leaving the notebook.

LOG_PATH = "/tmp/voxtral_server.log"
TAIL_LINES = 60  # adjust as needed

try:
    with open(LOG_PATH, encoding="utf-8") as fh:
        lines = fh.readlines()
    total = len(lines)
    shown = lines[-TAIL_LINES:]
    print(f"--- {LOG_PATH} (showing last {len(shown)} of {total} lines) ---")
    print("".join(shown))
except FileNotFoundError:
    print(f"{LOG_PATH} not found. Has the server been launched yet?")


## 3.2. Localhost Smoke Test (Colab-internal)
This cell sends a synthetic 1-second tone directly to `ws://localhost:8000/v1/realtime` **before going through ngrok**.

**Goal**: confirm the server's realtime inference path works end-to-end on Colab.
- ✅ Transcript returned (even empty string) → server realtime path is healthy
- ❌ Timeout or error → problem is in server inference, not the ngrok tunnel

> If this test passes but remote client times out, the bottleneck is ngrok or the network, not the server.

In [ ]:
# ── Localhost smoke test for realtime WebSocket ───────────────────────────────
# Sends a 1-second 440 Hz tone (PCM int16 @ 16kHz) to ws://localhost:8000/v1/realtime
# and waits up to 120 s for a transcript or error response.

import asyncio
import base64
import json
import time
import numpy as np
import websockets

SMOKE_URI = "ws://localhost:8000/v1/realtime"
SMOKE_TIMEOUT = 120  # seconds — inference on first run may be slow


def _make_tone_pcm(duration_s: float = 1.0, freq_hz: float = 440.0, sr: int = 16000) -> bytes:
    """Generate a pure-tone int16 PCM buffer."""
    t = np.linspace(0, duration_s, int(sr * duration_s), endpoint=False)
    wave = (np.sin(2 * np.pi * freq_hz * t) * 16000).astype(np.int16)
    return wave.tobytes()


async def _smoke_test():
    pcm = _make_tone_pcm(duration_s=1.0)
    print(f"[smoke] connecting to {SMOKE_URI} ...")
    t0 = time.time()
    async with websockets.connect(SMOKE_URI) as ws:
        print(f"[smoke] connected ({time.time()-t0:.2f}s)")

        # session.update
        await ws.send(json.dumps({
            "type": "session.update",
            "session": {"temperature": 0.0, "modalities": ["text"]}
        }))

        # send audio in one shot (no artificial delay — smoke test, not latency test)
        chunk_size = 3200  # 100ms @ 16kHz
        for i in range(0, len(pcm), chunk_size * 2):  # *2 because int16 = 2 bytes/sample
            chunk = pcm[i:i + chunk_size * 2]
            await ws.send(json.dumps({
                "type": "input_audio_buffer.append",
                "audio": base64.b64encode(chunk).decode()
            }))

        await ws.send(json.dumps({"type": "input_audio_buffer.commit"}))
        print(f"[smoke] audio committed ({time.time()-t0:.2f}s). Waiting up to {SMOKE_TIMEOUT}s ...")

        while True:
            try:
                raw = await asyncio.wait_for(ws.recv(), timeout=SMOKE_TIMEOUT)
            except asyncio.TimeoutError:
                print(f"[smoke] TIMEOUT after {SMOKE_TIMEOUT}s — no transcript received.")
                print("  → Check /tmp/voxtral_server.log for inference errors.")
                return
            msg = json.loads(raw)
            mtype = msg.get("type")
            if mtype == "response.audio_transcript.done":
                elapsed = time.time() - t0
                print(f"[smoke] SUCCESS in {elapsed:.2f}s — transcript: {msg.get('transcript')!r}")
                return
            elif mtype == "error":
                print(f"[smoke] ERROR from server: {msg}")
                print("  → Check /tmp/voxtral_server.log for traceback.")
                return
            elif mtype == "session.keepalive":
                print(f"[smoke] keepalive at {time.time()-t0:.1f}s (inference running ...)")
            else:
                print(f"[smoke] msg: {mtype}")


await _smoke_test()


## 4. Inference Client (Test Local)

In [ ]:
import asyncio
import base64
import json
import numpy as np
import librosa
import websockets

async def transcribe(audio_path, delay=480):
    """Send a local audio file to the server and return the transcript."""
    uri = "ws://localhost:8000/v1/realtime"
    # Load and convert to 16-bit PCM at 16kHz (what the server expects)
    audio, _ = librosa.load(audio_path, sr=16000, mono=True)
    audio_int16 = (audio * 32767).astype(np.int16)
    
    async with websockets.connect(uri) as ws:
        # 1. Configure session
        await ws.send(json.dumps({
            "type": "session.update",
            "session": {"transcription_delay_ms": delay, "temperature": 0.0}
        }))
        
        # 2. Stream audio in 100ms chunks
        chunk_samples = int(16000 * 0.1)  # 100ms window
        for i in range(0, len(audio_int16), chunk_samples):
            chunk = audio_int16[i:i+chunk_samples]
            await ws.send(json.dumps({
                "type": "input_audio_buffer.append",
                "audio": base64.b64encode(chunk.tobytes()).decode('utf-8')
            }))
            await asyncio.sleep(0.005)  # small yield
        
        # 3. Commit and wait for result
        await ws.send(json.dumps({"type": "input_audio_buffer.commit"}))
        
        async for message in ws:
            data = json.loads(message)
            if data.get("type") == "response.audio_transcript.done":
                return data.get('transcript', '')
            elif data.get("type") == "error":
                raise RuntimeError(data.get('error', {}).get('message', 'Unknown server error'))
    return "Transcription failed."

# Example usage
# transcript = await transcribe('audio/sample.mp3')
# print(transcript)